# 🚢 Day 01 — Titanic EDA
## 30 Days of Machine Learning

**Goal:** Explore the Titanic dataset — clean the data, uncover survival patterns, visualize insights, and build a baseline ML model.

**Topics Covered:**
- ✅ Data loading & first look
- ✅ Data cleaning & null handling
- ✅ Survival stats & analysis
- ✅ Charts & visualizations
- ✅ Bonus: Baseline ML model (Logistic Regression)

---

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

COLORS = {
    'survived': '#2ECC71',
    'died':     '#E74C3C',
    'neutral':  '#3498DB',
    'accent':   '#9B59B6'
}

print('✅ Libraries imported successfully!')

---
## 📂 Step 2 — Load Dataset

We'll load the Titanic dataset directly from a public URL (no Kaggle account needed!).

In [ ]:
# Load Titanic dataset from CSV
df = pd.read_csv("titanic.csv")

print(f"📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"
📋 Column Names:")
print(df.columns.tolist())
df.head(10)

In [ ]:
# Quick data types and non-null counts
df.info()

In [ ]:
# Basic statistics for numeric columns
df.describe().round(2)

---
## 🧹 Step 3 — Data Cleaning & Null Handling

In [ ]:
# ── Check missing values ──
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

missing_df = missing_df[missing_df['Missing Count'] > 0]
print('🔍 Missing Values:')
print(missing_df)

In [ ]:
# ── Visualize missing values ──
fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.barh(
    missing_df.index,
    missing_df['Missing %'],
    color=[COLORS['died'] if x > 20 else COLORS['neutral'] for x in missing_df['Missing %']],
    edgecolor='white', height=0.6
)

for bar, val in zip(bars, missing_df['Missing %']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Missing %', fontsize=12)
ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(0, 105)
plt.tight_layout()
plt.savefig('missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: missing_values.png')

In [ ]:
# ── Handle missing values ──
df_clean = df.copy()

# Age: fill with median (robust to outliers)
age_median = df_clean['age'].median()
df_clean['age'].fillna(age_median, inplace=True)
print(f'✅ Age: filled {df["age"].isnull().sum()} nulls with median ({age_median:.1f} years)')

# Embarked: fill with mode
emb_mode = df_clean['embarked'].mode()[0]
df_clean['embarked'].fillna(emb_mode, inplace=True)
print(f'✅ Embarked: filled {df["embarked"].isnull().sum()} nulls with mode ("{emb_mode}")')

# Deck: >75% missing — drop column
df_clean.drop(columns=['deck'], inplace=True)
print(f'✅ Deck: dropped column (77% missing)')

# Drop duplicate/redundant columns
cols_to_drop = ['embark_town', 'who', 'alive', 'class', 'adult_male', 'alone']
df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns], inplace=True)
print(f'✅ Dropped redundant derived columns')

print(f'\n📊 Clean dataset shape: {df_clean.shape}')
print(f'🔍 Remaining nulls: {df_clean.isnull().sum().sum()}')

In [ ]:
# ── Feature engineering for analysis ──

# Age groups
df_clean['age_group'] = pd.cut(
    df_clean['age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
)

# Family size
df_clean['family_size'] = df_clean['sibsp'] + df_clean['parch'] + 1
df_clean['is_alone'] = (df_clean['family_size'] == 1).astype(int)

# Fare categories
df_clean['fare_cat'] = pd.qcut(
    df_clean['fare'], q=4,
    labels=['Budget', 'Economy', 'Business', 'Luxury']
)

print('✅ New features created: age_group, family_size, is_alone, fare_cat')
df_clean.head()

---
## 📊 Step 4 — Survival Stats & Analysis

In [ ]:
# ── Overall survival rate ──
total = len(df_clean)
survived = df_clean['survived'].sum()
died = total - survived
surv_rate = survived / total * 100

print('=' * 45)
print('        📊 TITANIC SURVIVAL STATS')
print('=' * 45)
print(f'  Total Passengers : {total}')
print(f'  Survived         : {survived} ({surv_rate:.1f}%)')
print(f'  Did Not Survive  : {died} ({100 - surv_rate:.1f}%)')
print('=' * 45)

In [ ]:
# ── Survival by key categories ──
def surv_table(col, label):
    tbl = df_clean.groupby(col)['survived'].agg(['sum', 'count'])
    tbl.columns = ['Survived', 'Total']
    tbl['Rate %'] = (tbl['Survived'] / tbl['Total'] * 100).round(1)
    tbl['Died'] = tbl['Total'] - tbl['Survived']
    print(f'\n📌 Survival by {label}:')
    print(tbl.to_string())
    return tbl

surv_by_sex    = surv_table('sex', 'Sex')
surv_by_pclass = surv_table('pclass', 'Passenger Class')
surv_by_age    = surv_table('age_group', 'Age Group')
surv_by_alone  = surv_table('is_alone', 'Traveling Alone (1=Yes)')

---
## 📈 Step 5 — Visualizations

In [ ]:
# ── Plot 1: Overall Survival Donut Chart ──
fig, ax = plt.subplots(figsize=(7, 7))

sizes = [survived, died]
labels = [f'Survived\n{survived} ({surv_rate:.1f}%)', f'Did Not Survive\n{died} ({100-surv_rate:.1f}%)']
colors = [COLORS['survived'], COLORS['died']]
explode = (0.05, 0)

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    explode=explode, autopct='', startangle=90,
    wedgeprops=dict(width=0.6, edgecolor='white', linewidth=3)
)

for t in texts:
    t.set_fontsize(13)
    t.set_fontweight('bold')

ax.text(0, 0, f'{surv_rate:.0f}%\nSurvived', ha='center', va='center',
        fontsize=18, fontweight='bold', color='#2C3E50')

ax.set_title('Overall Titanic Survival Rate', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('survival_donut.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: survival_donut.png')

In [ ]:
# ── Plot 2: Survival by Sex & Passenger Class ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# By Sex
sex_data = df_clean.groupby(['sex', 'survived']).size().unstack()
sex_data.columns = ['Died', 'Survived']
sex_data[['Survived', 'Died']].plot(
    kind='bar', ax=axes[0],
    color=[COLORS['survived'], COLORS['died']],
    edgecolor='white', width=0.6
)
axes[0].set_title('Survival by Sex', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sex', fontsize=12)
axes[0].set_ylabel('Number of Passengers', fontsize=12)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0, fontsize=12)
axes[0].legend(fontsize=11)
for bar in axes[0].patches:
    axes[0].annotate(f'{int(bar.get_height())}',
                     (bar.get_x() + bar.get_width()/2, bar.get_height() + 3),
                     ha='center', fontsize=10, fontweight='bold')

# By Pclass
pclass_data = df_clean.groupby(['pclass', 'survived']).size().unstack()
pclass_data.columns = ['Died', 'Survived']
pclass_data[['Survived', 'Died']].plot(
    kind='bar', ax=axes[1],
    color=[COLORS['survived'], COLORS['died']],
    edgecolor='white', width=0.6
)
axes[1].set_title('Survival by Passenger Class', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Passenger Class', fontsize=12)
axes[1].set_ylabel('Number of Passengers', fontsize=12)
axes[1].set_xticklabels(['1st Class', '2nd Class', '3rd Class'], rotation=0, fontsize=11)
axes[1].legend(fontsize=11)
for bar in axes[1].patches:
    axes[1].annotate(f'{int(bar.get_height())}',
                     (bar.get_x() + bar.get_width()/2, bar.get_height() + 3),
                     ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Key Survival Factors', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('survival_sex_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: survival_sex_class.png')

In [ ]:
# ── Plot 3: Age Distribution by Survival ──
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(
    df_clean[df_clean['survived'] == 1]['age'],
    bins=30, alpha=0.7, color=COLORS['survived'],
    label='Survived', edgecolor='white'
)
ax.hist(
    df_clean[df_clean['survived'] == 0]['age'],
    bins=30, alpha=0.7, color=COLORS['died'],
    label='Did Not Survive', edgecolor='white'
)

ax.axvline(df_clean[df_clean['survived']==1]['age'].mean(), color='#27AE60',
           linestyle='--', linewidth=2, label=f'Survived Mean Age: {df_clean[df_clean["survived"]==1]["age"].mean():.1f}')
ax.axvline(df_clean[df_clean['survived']==0]['age'].mean(), color='#C0392B',
           linestyle='--', linewidth=2, label=f'Died Mean Age: {df_clean[df_clean["survived"]==0]["age"].mean():.1f}')

ax.set_title('Age Distribution by Survival', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Age', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: age_distribution.png')

In [ ]:
# ── Plot 4: Survival Rate Heatmap (Sex × Pclass) ──
fig, ax = plt.subplots(figsize=(8, 5))

heatmap_data = df_clean.pivot_table(
    values='survived', index='sex', columns='pclass', aggfunc='mean'
) * 100

sns.heatmap(
    heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn',
    linewidths=2, linecolor='white',
    annot_kws={'size': 14, 'weight': 'bold'},
    vmin=0, vmax=100, ax=ax,
    cbar_kws={'label': 'Survival Rate (%)'}
)

ax.set_title('Survival Rate (%) by Sex & Passenger Class', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Passenger Class', fontsize=12)
ax.set_ylabel('Sex', fontsize=12)
ax.set_xticklabels(['1st Class', '2nd Class', '3rd Class'], fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)
plt.tight_layout()
plt.savefig('survival_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: survival_heatmap.png')

In [ ]:
# ── Plot 5: Fare Distribution by Class (Box Plot) ──
fig, ax = plt.subplots(figsize=(10, 6))

class_labels = {1: '1st Class', 2: '2nd Class', 3: '3rd Class'}
df_box = df_clean.copy()
df_box['class_label'] = df_box['pclass'].map(class_labels)

sns.boxplot(
    data=df_box, x='class_label', y='fare',
    palette=['#3498DB', '#2ECC71', '#E74C3C'],
    order=['1st Class', '2nd Class', '3rd Class'],
    width=0.5, linewidth=1.5,
    flierprops=dict(marker='o', markersize=5, alpha=0.5),
    ax=ax
)

ax.set_title('Fare Distribution by Passenger Class', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Passenger Class', fontsize=12)
ax.set_ylabel('Fare (£)', fontsize=12)
plt.tight_layout()
plt.savefig('fare_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: fare_boxplot.png')

In [ ]:
# ── Plot 6: Correlation Heatmap ──
fig, ax = plt.subplots(figsize=(10, 7))

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_clean[numeric_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=1.5, linecolor='white',
    annot_kws={'size': 10}, square=True,
    vmin=-1, vmax=1, ax=ax
)

ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: correlation_matrix.png')

# Key insight
surv_corr = corr_matrix['survived'].drop('survived').abs().sort_values(ascending=False)
print('\n🔍 Features most correlated with Survival:')
print(surv_corr.round(3).to_string())

---
## 🤖 Step 6 — Bonus: Baseline ML Model (Logistic Regression)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)

# ── Prepare features ──
ml_df = df_clean.copy()

# Encode categorical
le = LabelEncoder()
ml_df['sex_enc'] = le.fit_transform(ml_df['sex'])          # female=0, male=1
ml_df['embarked_enc'] = le.fit_transform(ml_df['embarked'].astype(str))

features = ['pclass', 'sex_enc', 'age', 'sibsp', 'parch', 'fare',
            'embarked_enc', 'family_size', 'is_alone']

X = ml_df[features].fillna(ml_df[features].median())
y = ml_df['survived']

# ── Train/Test split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Scale features ──
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Training set:  {X_train.shape[0]} samples')
print(f'✅ Test set:      {X_test.shape[0]} samples')
print(f'✅ Features used: {features}')

In [ ]:
# ── Train Logistic Regression ──
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)

# ── Predictions ──
y_pred      = model.predict(X_test_sc)
y_pred_prob = model.predict_proba(X_test_sc)[:, 1]

# ── Metrics ──
acc     = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
cv_scores = cross_val_score(model, scaler.transform(X), y, cv=5, scoring='accuracy')

print('=' * 45)
print('     🤖 LOGISTIC REGRESSION RESULTS')
print('=' * 45)
print(f'  Test Accuracy   : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  ROC-AUC Score   : {roc_auc:.4f}')
print(f'  CV Score (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=' * 45)
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

In [ ]:
# ── Plot 7: Confusion Matrix + ROC Curve ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted\nDied', 'Predicted\nSurvived'],
    yticklabels=['Actual\nDied', 'Actual\nSurvived'],
    linewidths=2, linecolor='white',
    annot_kws={'size': 16, 'weight': 'bold'},
    ax=axes[0]
)
axes[0].set_title(f'Confusion Matrix\n(Accuracy: {acc*100:.1f}%)', fontsize=13, fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color=COLORS['neutral'], lw=2.5,
             label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='#95A5A6', lw=1.5, linestyle='--', label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color=COLORS['neutral'])
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11, loc='lower right')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: model_evaluation.png')

In [ ]:
# ── Plot 8: Feature Importance ──
fig, ax = plt.subplots(figsize=(10, 6))

coeff_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient')

colors = [COLORS['survived'] if c > 0 else COLORS['died'] for c in coeff_df['Coefficient']]

bars = ax.barh(coeff_df['Feature'], coeff_df['Coefficient'],
               color=colors, edgecolor='white', height=0.6)

ax.axvline(0, color='#555', linewidth=1.2, linestyle='-')
ax.set_title('Logistic Regression — Feature Coefficients\n(Green = Increases Survival, Red = Decreases Survival)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Coefficient Value', fontsize=12)

for bar, val in zip(bars, coeff_df['Coefficient']):
    ax.text(val + (0.02 if val >= 0 else -0.02),
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right',
            fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: feature_importance.png')

In [ ]:
# ── Save the model ──
import pickle

with open('titanic_model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'scaler': scaler, 'features': features}, f)

print('✅ Model saved: titanic_model.pkl')

# ── Quick prediction demo ──
sample = pd.DataFrame([{
    'pclass': 1, 'sex_enc': 0, 'age': 25, 'sibsp': 0, 'parch': 0,
    'fare': 80, 'embarked_enc': 0, 'family_size': 1, 'is_alone': 1
}])

sample_sc = scaler.transform(sample)
pred = model.predict(sample_sc)[0]
prob = model.predict_proba(sample_sc)[0][1]

print(f'\n🎯 Sample Prediction (1st class, female, 25yrs, solo):')
print(f'   Prediction : {"✅ SURVIVED" if pred == 1 else "❌ DID NOT SURVIVE"}')
print(f'   Probability: {prob*100:.1f}% chance of survival')